# EXP032b — Grid search for 1D cycle revivals

This notebook searches for full-state revivals of the discrete-time
quantum walk on $k$-cycles, using the coin family of Dukes (2014),
*Quantum state revivals in quantum walks on cycles*, Results in
Physics 4, 189–197. It is the companion to `EXP033.ipynb`, which
builds and verifies the 2D torus classification from a transcribed
1D catalogue.

## What this notebook is for

EXP033 transcribes the 1D revival catalogue from Dukes' tables and
verifies each entry at the operator level. That verification
confirms the transcribed entries are correct, but it does not
*search* for revivals — it checks a fixed list.

EXP032b is the search. It sweeps the coin parameters $(\rho, \delta)$
on a grid, evolves the walk, and reports every step $N$ at which the
state returns. Its purpose is threefold:

1. **Confirm the Dukes entries.** Every tabulated $(\rho, \delta, N)$
   in Dukes' Tables 2–5 should appear in the scan output. This is a
   cross-check on the transcription in EXP033.
2. **Find any revivals Dukes did not tabulate.** The scan compares
   every detected revival against the `DUKES_EXACT` table and flags
   anything not present. This is how the $k = 2$, $\rho = 1/2$,
   $\delta \in \{\pi/2, 3\pi/2\}$ correction was found.
3. **Check the negative cases.** For $k \notin \{2, 3, 4, 5, 6, 8, 10\}$,
   the scan reports whether any revival is detected up to $N = 60$.
   A null result is evidence — but see the scope caveat below — that
   these cycle lengths admit no revival.

## What the scan does

For each $k$ in the $k$-set and each $(\rho, \delta)$ on the parameter
grid, the notebook:

1. Builds the walk operator $U_k(\rho, \delta)$ and the initial state
   $|0, \text{up}\rangle$.
2. Evolves the state step by step for $N = 1, \ldots, N_{\max}$.
3. At each step, classifies the state as one of:
   - **full** — the state vector equals $c \cdot \psi_0$ with
     $|c| = 1$ (a *state-level projective* revival: this particular
     initial state returns up to a phase);
   - **pos_only** — the position distribution returns but the coin
     state does not;
   - **coin_only** — the coin state returns but the position does not;
   - **coin_triv** — the coin state returns and the walker is
     localized (a trivial match, not a revival).
4. Records the full revival steps for each $(\rho, \delta)$.

## Scope caveats

Two caveats matter when reading the output, and they should be kept
in mind when citing results from this notebook in the paper.

**The criterion is state-level, not operator-level.** The `full` test
is `same_up_to_phase(psi, psi0)`, which checks whether *the
particular initial state* $|0, \text{up}\rangle$ returns up to a
phase. It does not check $U^N = I$ (every state returns with no
phase), nor does it check $U^N = e^{i\phi}I$ (every state returns
up to a phase). A state-level revival is weaker than either. The
exact operator-level criterion is checked in EXP033 and in the
`--exhaustive` mode of this notebook.

**The parameter grid is finite.** By default the scan uses
$201$ points in $\rho \in [0, 1]$ and a fixed set of ten $\delta$
values. A revival that does not fall within $\sim 0.005$ of a $\rho$
grid point, or within $\sim 10^{-6}$ of one of the ten $\delta$
values, will not be detected. The scan is therefore a *grid search*,
not an exhaustive search. For an operator-level exhaustive check on
a dense $(\rho, \delta)$ grid, use the `--exhaustive` mode
(see below).

## How to run it

The notebook writes `EXP032b.py` to disk (via `%%writefile`) and
then runs it with command-line arguments. Two modes:

**Grid-search mode** (the default, state-level criterion):

```bash
# Single delta
python EXP032b.py --verbose 0 --rho-steps 201 --N 60 --csv dukes_scan.csv

# Loop over the ten standard delta values
python EXP032b.py --delta-sweep --csv sweep.csv

# Just one k
python EXP032b.py --k 4 --beta 1.5707963 --N 20 --verbose 0

Use Dukes published rho values to verify revivals (delta = 0)

In [ ]:
%%writefile EXP032b.py
#!/usr/bin/env python3
# ============================================================
#  EXP032b.py  (v5)
#
#  Sweep (rho, delta) for discrete quantum walks on k-cycles,
#  using the coin of Dukes (2014), "Quantum state revivals in
#  quantum walks on cycles".
#
#  v5 changes vs v4:
#    - Adds --exhaustive mode: dense grid over (rho, delta)
#      with operator-level exact criterion (U^N = I) and
#      projective criterion (U^N = e^{i phi} I).  Reports
#      the worst residual per k so the negative result is
#      quantified, not just stated.
#    - Adds --eig-residual flag: use eigenvalue residual
#      (max_j |lambda_j^N - lambda^N_mean|) instead of
#      matrix_power for the exact check.  Required for
#      N up to 60; matrix_power is fine for small N.
#    - Adds --rho-fine / --delta-fine: denser grids for the
#      exhaustive mode (default 1001 x 360, ~360k points per k).
#    - Preserves v4 behaviour for --delta-sweep and single-delta
#      runs (state-level same_up_to_phase, as before).
#    - Adds --k-all: 2..16 plus 20, 24 (the v4 default), or
#      any explicit comma-list.
# ============================================================

import numpy as np
import csv
import argparse
import sys
import time
from math import sqrt, pi, gcd
from fractions import Fraction

# ------------------------------------------------------------
#  Configuration
# ------------------------------------------------------------

K_SET = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 20, 24]

RHO_MIN   = 0.0
RHO_MAX   = 1.0
RHO_STEPS = 201

# Exhaustive-mode defaults (dense enough to be a real check,
# coarse enough to run in minutes not hours)
RHO_FINE_STEPS   = 1001       # rho in {0, 1/1000, ..., 1}
DELTA_FINE_STEPS = 360        # delta in {0, 2pi/360, ..., 2pi}

N_STEPS   = 60
ALPHA     = 0.0
BETA      = 0.0
POS0      = 0
SPIN0     = 'up'

TOL_PHASE = 1e-9
TOL_POS   = 1e-7
TOL_COIN  = 1e-7
TOL_AMP   = 1e-12

# Exact criterion tolerance.  Tight enough to reject false
# positives; loose enough to survive numerical roundoff at
# N=60 on 2k x 2k matrices.
TOL_EXACT = 1e-9

# ------------------------------------------------------------
#  Dukes exact revival table (v4, unchanged)
#  (k, rho, delta, N_expected, source)
# ------------------------------------------------------------

DUKES_EXACT = [
    (2, 0.5, 0.0, 2, "Table 2"),

    (3, 2/3,              0.0,        8,  "Table 3"),
    (3, (5 - sqrt(5))/6,  0.0,        10, "Table 3"),
    (3, 1/3,              0.0,        12, "Table 3"),
    (3, 1/3,              2*pi/3,     12, "Table 3"),
    (3, 1/3,              4*pi/3,     12, "Table 3"),

    (4, 3/4,              0.0,        6,  "Table 4"),
    (4, 1/2,              0.0,        8,  "Table 4"),
    (4, 1/2,              pi,         8,  "Table 4"),
    (4, (5 - sqrt(5))/8,  0.0,        10, "Table 4"),
    (4, (5 + sqrt(5))/8,  0.0,        10, "Table 4"),
    (4, 1/4,              0.0,        12, "Table 4"),
    (4, 1/4,              pi,         12, "Table 4"),
    (4, 3/4,              pi,         12, "Table 4"),
    (4, (2 - sqrt(3))/2,  pi/2,       12, "Table 4"),
    (4, (2 - sqrt(3))/2,  3*pi/2,     12, "Table 4"),

    (5,  (5 - sqrt(5))/10, 0.0,     60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 0.0,     60, "Table 5"),
    (5,  (5 - sqrt(5))/10, 2*pi/5,  60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 2*pi/5,  60, "Table 5"),
    (5,  (5 - sqrt(5))/10, 4*pi/5,  60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 4*pi/5,  60, "Table 5"),
    (5,  (5 - sqrt(5))/10, 6*pi/5,  60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 6*pi/5,  60, "Table 5"),
    (5,  (5 - sqrt(5))/10, 8*pi/5,  60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 8*pi/5,  60, "Table 5"),
    (10, (5 - sqrt(5))/10, 0.0,     60, "Table 5"),
    (10, (5 + sqrt(5))/10, 0.0,     60, "Table 5"),
    (10, (5 - sqrt(5))/10, 2*pi/5,  60, "Table 5"),
    (10, (5 + sqrt(5))/10, 2*pi/5,  60, "Table 5"),
    (10, (5 - sqrt(5))/10, 4*pi/5,  60, "Table 5"),
    (10, (5 + sqrt(5))/10, 4*pi/5,  60, "Table 5"),
    (10, (5 - sqrt(5))/10, 6*pi/5,  60, "Table 5"),
    (10, (5 + sqrt(5))/10, 6*pi/5,  60, "Table 5"),
    (10, (5 - sqrt(5))/10, 8*pi/5,  60, "Table 5"),
    (10, (5 + sqrt(5))/10, 8*pi/5,  60, "Table 5"),

    (8, 1/2, 0.0,    24, "Table 5"),
    (8, 1/2, pi/2,   24, "Table 5"),
    (8, 1/2, pi,     24, "Table 5"),
    (8, 1/2, 3*pi/2, 24, "Table 5"),
]

DELTA_SWEEP = [
    (0.0,      "0"),
    (2*pi/3,   "2pi/3"),
    (pi/2,     "pi/2"),
    (pi,       "pi"),
    (4*pi/3,   "4pi/3"),
    (3*pi/2,   "3pi/2"),
    (2*pi/5,   "2pi/5"),
    (4*pi/5,   "4pi/5"),
    (6*pi/5,   "6pi/5"),
    (8*pi/5,   "8pi/5"),
]

# ------------------------------------------------------------
#  Core operators
# ------------------------------------------------------------

def coin_operator(rho, alpha=0.0, beta=0.0):
    return np.array([
        [sqrt(rho),                    sqrt(1-rho)*np.exp(1j*alpha)],
        [sqrt(1-rho)*np.exp(1j*beta), -sqrt(rho)*np.exp(1j*(alpha+beta))]
    ], dtype=complex)

def shift_operator(k):
    dim = 2*k
    S = np.zeros((dim, dim), dtype=complex)
    for i in range(k):
        S[((i-1) % k)*2,     i*2]     = 1.0
        S[((i+1) % k)*2 + 1, i*2 + 1] = 1.0
    return S

def walk_operator(k, rho, alpha=0.0, beta=0.0):
    return shift_operator(k) @ np.kron(np.eye(k, dtype=complex),
                                       coin_operator(rho, alpha, beta))

def initial_state(k, pos=0, spin='up'):
    psi = np.zeros(2*k, dtype=complex)
    psi[pos*2 + (0 if spin == 'up' else 1)] = 1.0
    return psi

# ------------------------------------------------------------
#  Eigenvalue residuals (exact and projective)
# ------------------------------------------------------------

def exact_residual_eig(U, N):
    """Exact criterion U^N = I via eigenvalues.

    max_j |lambda_j^N - 1|.  Vanishes iff U^N = I.

    Uses eigenvalue exponentiation rather than matrix_power
    because matrix_power accumulates error at N ~ 60 on
    matrices of size 2k x 2k (k up to 16 -> 32 x 32).
    """
    eigs = np.linalg.eigvals(U)
    z = eigs ** N
    return float(np.max(np.abs(z - 1.0)))

def projective_residual_eig(U, N):
    """Projective criterion U^N = e^{i phi} I via eigenvalues.

    max_j |lambda_j^N - mean(lambda^N)|.  Vanishes iff all
    lambda_j^N share a common value (i.e. U^N = e^{i phi} I).
    """
    eigs = np.linalg.eigvals(U)
    z = eigs ** N
    z_mean = z.mean()
    if abs(z_mean) < 1e-15:
        return float(np.max(np.abs(z)))
    return float(np.max(np.abs(z - z_mean)))

def exact_residual_matrixt(U, N):
    """Reference exact residual via matrix_power (slow, but
    agrees with exact_residual_eig to numerical precision
    for N small enough that roundoff is negligible)."""
    UN = np.linalg.matrix_power(U, N)
    return float(np.linalg.norm(UN - np.eye(UN.shape[0])))

# ------------------------------------------------------------
#  State-level comparison (v4, unchanged)
# ------------------------------------------------------------

def same_up_to_phase(a, b, tol=TOL_PHASE):
    if np.linalg.norm(a) < tol or np.linalg.norm(b) < tol:
        return False
    idx = int(np.argmax(np.abs(b)))
    phase = a[idx] / b[idx]
    if abs(abs(phase) - 1.0) > tol:
        return False
    return np.allclose(a, phase * b, atol=tol)

def position_distribution(psi, k):
    p = np.abs(psi)**2
    return p[0::2] + p[1::2]

def coin_state(psi, k):
    psi_mat = psi.reshape(k, 2)
    return psi_mat.conj().T @ psi_mat

def same_position_distribution(pa, pb, tol=TOL_POS):
    return np.allclose(pa, pb, atol=tol)

def same_coin_state(ra, rb, tol=TOL_COIN):
    return np.allclose(ra, rb, atol=tol)

def is_localized(psi, k, tol=TOL_AMP):
    p = position_distribution(psi, k)
    return int(np.sum(p > tol)) == 1

# ------------------------------------------------------------
#  v4 scan_one_rho (state-level, unchanged)
# ------------------------------------------------------------

def scan_one_rho(k, rho, N, alpha, beta, pos0, spin0):
    U    = walk_operator(k, rho, alpha, beta)
    psi0 = initial_state(k, pos0, spin0)
    psi  = psi0.copy()

    out = dict(full=[], pos_only=[], coin_only=[],
               both_phase=[], coin_triv=[])

    for step in range(1, N+1):
        psi = U @ psi

        full = same_up_to_phase(psi, psi0)
        pos  = same_position_distribution(
            position_distribution(psi, k),
            position_distribution(psi0, k))
        coin = same_coin_state(coin_state(psi, k),
                               coin_state(psi0, k))
        loc  = is_localized(psi, k)

        if full:
            out['full'].append(step)
        elif pos and coin:
            out['both_phase'].append(step)
        elif pos:
            out['pos_only'].append(step)
        elif coin:
            if loc:
                out['coin_triv'].append(step)
            else:
                out['coin_only'].append(step)

    return out

# ------------------------------------------------------------
#  EXHAUSTIVE scan (v5, operator-level)
#
#  For a given (k, rho, delta), compute:
#    - the minimal exact period N_exact (smallest N with
#      lambda_j^N = 1 for all j, up to TOL_EXACT), searching
#      N = 1..N_max
#    - the minimal projective period N_proj (smallest N with
#      lambda_j^N = e^{i phi} for a common phi)
#    - the worst exact residual over N = 1..N_max, so the
#      negative result is quantified
# ------------------------------------------------------------

def scan_exhaustive_one_point(k, rho, delta, N_max, tol=TOL_EXACT):
    """Operator-level exact/projective scan at a single (rho, delta).

    Returns dict with:
      n_exact      : smallest N in [1, N_max] with U^N = I, or None
      n_proj       : smallest N with U^N = e^{i phi} I, or None
      phi_proj     : the phase e^{i phi} at N = n_proj
      worst_exact  : max over N of exact_residual_eig(U, N)
      worst_proj   : max over N of projective_residual_eig(U, N)

    U is diagonalised ONCE.  The N loop only does elementwise
    exponentiation (cheap: O(dim) per N, not O(dim^3)).
    """
    U = walk_operator(k, rho, 0.0, delta)
    eigs = np.linalg.eigvals(U)

    n_exact = None
    n_proj = None
    phi_proj = None
    worst_exact = 0.0
    worst_proj = 0.0

    for N in range(1, N_max + 1):
        z = eigs ** N

        # Exact: all z_j == 1
        e_res = float(np.max(np.abs(z - 1.0)))
        if e_res > worst_exact:
            worst_exact = e_res
        if n_exact is None and e_res < tol:
            n_exact = N

        # Projective: all z_j share a common value
        z_mean = z.mean()
        if abs(z_mean) > 1e-15:
            p_res = float(np.max(np.abs(z - z_mean)))
            if p_res > worst_proj:
                worst_proj = p_res
            if n_proj is None and p_res < tol:
                n_proj = N
                phi_proj = z_mean / abs(z_mean)

        # Short-circuit: if both found, no need to continue
        # (worst_exact / worst_proj are only used for the
        # negative result, and we already have positive results
        # for this point)
        if n_exact is not None and n_proj is not None:
            break

    return dict(n_exact=n_exact, n_proj=n_proj, phi_proj=phi_proj,
                worst_exact=worst_exact, worst_proj=worst_proj)

# ------------------------------------------------------------
#  Build rho / delta grids
# ------------------------------------------------------------

def build_rho_grid(k, rho_min, rho_max, rho_steps,
                   delta, include_exact=True, exact_tol=1e-6):
    grid = list(np.linspace(rho_min, rho_max, rho_steps))
    if include_exact:
        related = {k}
        if k % 2 == 0 and (k // 2) % 2 == 1:
            related.add(k // 2)
        if k % 2 == 1:
            related.add(2 * k)
        for (kk, rr, dd, _, _) in DUKES_EXACT:
            if kk not in related:
                continue
            if abs(dd - delta) > exact_tol:
                continue
            if rho_min - exact_tol <= rr <= rho_max + exact_tol:
                grid.append(rr)
    grid = sorted(set(round(x, 12) for x in grid))
    return grid

def build_delta_grid(delta_steps):
    """Dense delta grid in [0, 2pi).  Includes the standard
    Dukes delta values exactly (0, pi/2, pi, 3pi/2, 2pi/5,
    2pi/3, 4pi/5, 4pi/3, 6pi/5, 8pi/5)."""
    grid = list(np.linspace(0.0, 2*pi, delta_steps, endpoint=False))
    for (d, _) in DELTA_SWEEP:
        grid.append(d)
    grid = sorted(set(round(x, 12) for x in grid))
    return grid

# ------------------------------------------------------------
#  Dukes lookup (v4, unchanged)
# ------------------------------------------------------------

def is_dukes_exact(k, rho, delta, N1,
                   rho_tol=1e-4, delta_tol=1e-4, N_tol=1):
    for (kk, rr, dd, NN, src) in DUKES_EXACT:
        if kk != k:           continue
        if abs(rr - rho) > rho_tol:   continue
        if abs(dd - delta) > delta_tol: continue
        if N1 is not None and abs(NN - N1) > N_tol: continue
        return src
    return None

def snap_delta(delta, tol=1e-6):
    exact_deltas = sorted({dd for (_, _, dd, _, _) in DUKES_EXACT})
    for ed in exact_deltas:
        if abs(delta - ed) < tol:
            return ed
    return delta

# ------------------------------------------------------------
#  v4 sweep driver (state-level, unchanged)
# ------------------------------------------------------------

def sweep(k_list, rho_min, rho_max, rho_steps,
          N, alpha, beta, pos0, spin0, verbose=1):
    delta   = alpha + beta
    records = []

    for k in k_list:
        rho_grid = build_rho_grid(k, rho_min, rho_max, rho_steps, delta)

        if verbose >= 1:
            print(f"\n{'='*70}")
            print(f"  k = {k}   delta = {delta:.6f}   "
                  f"({len(rho_grid)} rho points, N_max = {N})")
            print(f"{'='*70}")

        for rho in rho_grid:
            r = scan_one_rho(k, rho, N, alpha, beta, pos0, spin0)
            rec = dict(k=k, rho=float(rho), delta=float(delta),
                       full=r['full'],
                       pos_only=r['pos_only'],
                       coin_only=r['coin_only'],
                       both_phase=r['both_phase'],
                       coin_triv=r['coin_triv'])
            records.append(rec)

            if verbose >= 2:
                def s(v): return "-" if not v else ",".join(map(str, v))
                print(f"  {rho:8.4f}  full={s(r['full']):<30} "
                      f"pos={s(r['pos_only']):<10} "
                      f"coin={s(r['coin_only']):<10} "
                      f"both={s(r['both_phase']):<10}")

        if verbose >= 1:
            fulls = [r for r in records if r['k'] == k and r['full']]
            trivp = [r for r in records if r['k'] == k and r['coin_triv']]
            print(f"  -> {len(fulls)} rho values give a FULL revival")
            for rec in fulls[:12]:
                print(f"       rho={rec['rho']:.6f}  N={rec['full']}")
            if len(fulls) > 12:
                print(f"       ... (+{len(fulls)-12} more)")
            if trivp:
                print(f"  -> {len(trivp)} rho values give trivial coin "
                      f"matches (localized)")

    return records

# ------------------------------------------------------------
#  EXHAUSTIVE driver (v5)
#
#  Enumerates (rho, delta) on a dense grid and reports:
#    - every exact revival found (k, rho, delta, N_exact)
#    - every projective revival found (k, rho, delta, N_proj)
#    - for k with no exact revival, the worst exact residual
#      observed across the grid (the "sensitivity witness")
# ------------------------------------------------------------

def exhaustive_sweep(k_list, rho_steps, delta_steps, N_max,
                     rho_min=RHO_MIN, rho_max=RHO_MAX,
                     verbose=True):
    delta_grid = build_delta_grid(delta_steps)
    rho_grid_base = np.linspace(rho_min, rho_max, rho_steps)

    print("="*78)
    print("EXHAUSTIVE (rho, delta) SWEEP — operator-level criterion")
    print("="*78)
    print(f"  k values          : {k_list}")
    print(f"  rho grid          : {rho_steps} points in [{rho_min}, {rho_max}]")
    print(f"  delta grid        : {delta_steps} points in [0, 2pi)")
    print(f"  N max             : {N_max}")
    print(f"  exact tolerance   : {TOL_EXACT:.1e}")
    print(f"  points per k      : {rho_steps * len(delta_grid):,}")
    print()

    results = {}

    for k in k_list:
        rho_grid = build_rho_grid(k, rho_min, rho_max, rho_steps,
                                  delta=0.0, include_exact=True)
        # Note: build_rho_grid injects Dukes values for the
        # delta given.  Since we sweep delta anyway, we don't
        # inject per-delta — the Dukes rho values will appear
        # in the rho grid only if they happen to be a grid point.
        # To ensure Dukes values are always tested, add them
        # for every delta in the grid:
        dukes_rhos = sorted({rr for (kk, rr, _, _, _) in DUKES_EXACT
                             if kk == k})
        for rr in dukes_rhos:
            if rho_min - 1e-9 <= rr <= rho_max + 1e-9:
                rho_grid = sorted(set(rho_grid + [rr]))

        n_points = len(rho_grid) * len(delta_grid)
        t0 = time.time()
        exact_hits = []
        proj_hits = []
        worst_exact = 0.0
        worst_exact_at = None
        n_done = 0

        for rho in rho_grid:
            for delta in delta_grid:
                r = scan_exhaustive_one_point(k, rho, delta, N_max)

                if r['n_exact'] is not None:
                    exact_hits.append((rho, delta, r['n_exact']))
                if r['n_proj'] is not None:
                    proj_hits.append((rho, delta, r['n_proj'],
                                      r['phi_proj']))

                if r['worst_exact'] > worst_exact:
                    worst_exact = r['worst_exact']
                    worst_exact_at = (rho, delta)

                n_done += 1
                if verbose and n_done % max(1, n_points // 10) == 0:
                    elapsed = time.time() - t0
                    pct = 100 * n_done / n_points
                    eta = elapsed * (n_points - n_done) / max(n_done, 1)
                    print(f"  k={k:>2}  {n_done:>6}/{n_points}  "
                          f"({pct:5.1f}%)  "
                          f"exact={len(exact_hits)}  "
                          f"proj={len(proj_hits)}  "
                          f"ETA {eta:5.0f}s", end="\r")
        if verbose:
            print()

        elapsed = time.time() - t0

        # --- refinement pass: snap each candidate and confirm by matrix_power ---
        refined = []
        unconfirmed = []
        for (rho, delta, N) in exact_hits:
            r = refine_candidate(k, rho, delta, N)
            if r['confirmed']:
                refined.append(dict(rho=rho, delta=delta, N=N,
                                    rho_snap=r['rho_snap'],
                                    delta_snap=r['delta_snap'],
                                    eig_resid=r['eig_resid'],
                                    mat_resid=r['mat_resid']))
            else:
                unconfirmed.append(dict(rho=rho, delta=delta, N=N,
                                        rho_snap=r['rho_snap'],
                                        delta_snap=r['delta_snap'],
                                        eig_resid=r['eig_resid'],
                                        mat_resid=r['mat_resid']))

        results[k] = dict(
            rho_grid_size=len(rho_grid),
            delta_grid_size=len(delta_grid),
            n_points=n_points,
            elapsed=elapsed,
            exact_hits=exact_hits,       # raw candidates (all that passed tolerance)
            refined_hits=refined,        # confirmed by matrix_power
            unconfirmed=unconfirmed,     # passed grid tolerance but failed refinement
            proj_hits=proj_hits,
            worst_exact=worst_exact,
            worst_exact_at=worst_exact_at,
        )

        print(f"  k={k:>2}:  "
              f"candidates={len(exact_hits):>4}  "
              f"refined={len(refined):>4}  "
              f"unconfirmed={len(unconfirmed):>3}  "
              f"proj={len(proj_hits):>4}  "
              f"worst_exact={worst_exact:.2e}  "
              f"({elapsed:.1f}s)")

    return results, rho_grid_base, delta_grid

# ------------------------------------------------------------
#  Report for exhaustive mode
# ------------------------------------------------------------

def classify_hit(rho, delta, k, N, tol=1e-6):
    """Classify an (rho, delta, N) exact revival.

    Returns:
      'trivial' -- degenerate coin (permutation at delta in {0, pi},
                   or any delta at rho in {0, 1}), so the walk
                   revives trivially for many parameter values
      'dukes'   -- matches a DUKES_EXACT entry
      'new'     -- neither (candidate for a new revival)
    """
    # Trivial delta: coin is a permutation
    if abs(delta) < tol or abs(delta - pi) < tol or abs(delta - 2*pi) < tol:
        return 'trivial'

    # Trivial rho: coin is a permutation
    #   rho = 0 -> [[0, e^{i alpha}], [e^{i beta}, 0]]
    #   rho = 1 -> [[1, 0], [0, -e^{i(alpha+beta)}]]
    # In both cases the walk reduces to a permutation of the position
    # space with an overall phase, and revives trivially.
    if abs(rho) < tol or abs(rho - 1.0) < tol:
        return 'trivial'

    src = is_dukes_exact(k, rho, delta, N)
    if src is not None:
        return 'dukes'
    return 'new'

def print_exhaustive_report(results, k_list, out_txt=None):
    lines = []
    def p(s=""):
        print(s)
        lines.append(s)

    p()
    p("="*78)
    p("EXHAUSTIVE SWEEP — SUMMARY")
    p("="*78)

    for k in k_list:
        r = results[k]
        p()
        p(f"k = {k}")
        p(f"  grid: {r['rho_grid_size']} rho x {r['delta_grid_size']} delta "
          f"= {r['n_points']:,} points")

        # --- three-way split of refined hits ---
        trivial_hits = []
        dukes_hits   = []
        new_hits     = []
        for (rho, delta, N) in r['refined_hits']:
            kind = classify_hit(rho, delta, k, N)
            if kind == 'trivial':
                trivial_hits.append((rho, delta, N))
            elif kind == 'dukes':
                dukes_hits.append((rho, delta, N))
            else:
                new_hits.append((rho, delta, N))

        p(f"  confirmed revivals found: {len(r['refined_hits'])}  "
          f"({len(trivial_hits)} trivial, "
          f"{len(dukes_hits)} Dukes, "
          f"{len(new_hits)} new)")

        # --- warn about unconfirmed candidates ---
        if r['unconfirmed']:
            p(f"  WARNING: {len(r['unconfirmed'])} unconfirmed candidates "
              f"(passed grid tolerance, failed refinement)")
            for c in r['unconfirmed'][:5]:
                p(f"    rho={c['rho']:.6f}  delta={c['delta']:.6f}  "
                  f"N={c['N']}  "
                  f"eig={c['eig_resid']:.2e}  mat={c['mat_resid']:.2e}")
            if len(r['unconfirmed']) > 5:
                p(f"    ... (+{len(r['unconfirmed'])-5} more)")

        # --- detail listing ---
        if new_hits:
            by_n = {}
            for (rho, delta, N) in new_hits:
                by_n.setdefault(N, []).append((rho, delta))
            p(f"  NEW exact revivals:")
            for N in sorted(by_n):
                pts = by_n[N]
                p(f"    N = {N:>3}:  {len(pts)} point(s)")
                for (rho, delta) in pts[:8]:
                    p(f"      rho={rho:.6f}  delta={delta:.6f}")
                if len(pts) > 8:
                    p(f"      ... (+{len(pts)-8} more)")

        elif dukes_hits:
            by_n = {}
            for (rho, delta, N) in dukes_hits:
                by_n.setdefault(N, []).append((rho, delta))
            p(f"  Dukes-tabulated exact revivals:")
            for N in sorted(by_n):
                pts = by_n[N]
                p(f"    N = {N:>3}:  {len(pts)} point(s)")
                for (rho, delta) in pts[:6]:
                    p(f"      rho={rho:.6f}  delta={delta:.6f}")
                if len(pts) > 6:
                    p(f"      ... (+{len(pts)-6} more)")

        else:
            p(f"  (no Dukes-tabulated or new exact revivals on this grid)")

        # --- sensitivity witness ---
        p(f"  worst exact residual over grid: "
          f"{r['worst_exact']:.2e}")
        if r['worst_exact_at']:
            rho_w, delta_w = r['worst_exact_at']
            p(f"    at rho={rho_w:.6f}, delta={delta_w:.6f}")

        p(f"  projective revivals found: {len(r['proj_hits'])}")

    if out_txt:
        with open(out_txt, 'w') as f:
            f.write("\n".join(lines))
        print(f"\nReport written to {out_txt}")

# ------------------------------------------------------------
#  v4 compact summary (unchanged)
# ------------------------------------------------------------

def _format_harmonics(harms):
    if not harms:
        return "-"
    if len(harms) <= 3:
        return ", ".join(str(x) for x in harms)
    d = harms[1] - harms[0]
    is_ap = all(harms[i+1] - harms[i] == d for i in range(len(harms)-1))
    if is_ap:
        return f"{harms[0]}, {harms[0]+d}, ..., {harms[-1]}"
    return f"{harms[0]}, {harms[1]}, ..., {harms[-1]}"

def print_compact_summary(records, k_list, skip_endpoints=True,
                          flag_new=False):
    print("\n" + "#"*70)
    print("#  COMPACT SUMMARY — FULL revivals (interior rho only)")
    if flag_new:
        print("#  (*)  = not in Dukes' tables")
    print("#"*70)

    for k in k_list:
        if k == 2:
            print(f"\nk = 2")
            print("  (continuous family: all rho in [0,1] give "
                  "N1 = 2 at delta = 0)")
            continue

        recs = [r for r in records if r['k'] == k and r['full']]
        if skip_endpoints:
            recs = [r for r in recs
                    if 1e-6 < r['rho'] < 1 - 1e-6]

        print(f"\nk = {k}")
        if not recs:
            print("  (no interior full revivals)")
            continue

        print(f"  {'':1} {'rho':>12}  {'N1':>4}  {'harmonics':>24}")
        for r in sorted(recs, key=lambda x: x['rho']):
            n1 = r['full'][0]
            harms = [n for n in r['full'] if n != n1]
            marker = " "
            if flag_new:
                src = is_dukes_exact(k, r['rho'], r['delta'], n1)
                marker = " " if src else "*"
            print(f"  {marker} {r['rho']:12.6f}  {n1:>4}  "
                  f"{_format_harmonics(harms):>24}")

#
# `fine scan checks
#

def snap_rho(rho, max_denom=30, tol=1e-3):
    """Snap rho to the nearest rational with small denominator,
    or return rho unchanged if no nearby rational exists."""
    frac = Fraction(rho).limit_denominator(max_denom)
    if abs(float(frac) - rho) < tol:
        return float(frac)
    return rho

def snap_delta(delta, max_denom=60, tol=1e-3):
    """Snap delta to the nearest rational multiple of 2*pi."""
    frac = Fraction(delta / (2*pi)).limit_denominator(max_denom)
    snapped = float(frac) * 2 * pi
    if abs(snapped - delta) < tol:
        return snapped
    return delta

def refine_candidate(k, rho, delta, N_exact, tol_eig=1e-12, tol_mat=1e-10):
    """Refine a candidate revival: snap parameters, recheck eigenvalue
    residual at tight tolerance, then confirm with matrix_power.

    Returns dict with:
      confirmed   : True if the matrix-power check passes
      rho_snap    : the snapped rho
      delta_snap  : the snapped delta
      eig_resid   : the eigenvalue residual at the snapped params
      mat_resid   : the matrix-power residual at the snapped params
    """
    rho_s   = snap_rho(rho)
    delta_s = snap_delta(delta)

    U = walk_operator(k, rho_s, 0.0, delta_s)

    # Step 2: tight eigenvalue residual
    eigs = np.linalg.eigvals(U)
    z = eigs ** N_exact
    eig_resid = float(np.max(np.abs(z - 1.0)))

    # Step 3: matrix-power confirmation
    UN = np.linalg.matrix_power(U, N_exact)
    mat_resid = float(np.linalg.norm(UN - np.eye(UN.shape[0])))

    confirmed = (eig_resid < tol_eig) and (mat_resid < tol_mat)

    return dict(confirmed=confirmed,
                rho_snap=rho_s, delta_snap=delta_s,
                eig_resid=eig_resid, mat_resid=mat_resid)

# ------------------------------------------------------------
#  CSV export (v4, unchanged)
# ------------------------------------------------------------

def write_csv(records, filename):
    with open(filename, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['k', 'rho', 'delta', 'full', 'pos_only',
                    'coin_only', 'both_phase', 'coin_triv'])
        for r in records:
            def j(v): return ";".join(map(str, v)) if v else ""
            w.writerow([r['k'], f"{r['rho']:.10f}", f"{r['delta']:.10f}",
                        j(r['full']), j(r['pos_only']),
                        j(r['coin_only']), j(r['both_phase']),
                        j(r['coin_triv'])])

# ------------------------------------------------------------
#  Main
# ------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(
        description="Sweep rho/delta for Dukes quantum walk revivals "
                    "on k-cycles.")
    parser.add_argument('--k', type=str, default=None,
                        help='comma-separated k values (default: K_SET)')
    parser.add_argument('--k-all', action='store_true',
                        help='explicit K_SET (2..16, 20, 24)')
    parser.add_argument('--rho-min',   type=float, default=RHO_MIN)
    parser.add_argument('--rho-max',   type=float, default=RHO_MAX)
    parser.add_argument('--rho-steps', type=int,   default=RHO_STEPS)
    parser.add_argument('--N',         type=int,   default=N_STEPS)
    parser.add_argument('--alpha',     type=float, default=ALPHA)
    parser.add_argument('--beta',      type=float, default=BETA)
    parser.add_argument('--pos0',      type=int,   default=POS0)
    parser.add_argument('--spin0', choices=['up', 'down'], default=SPIN0)
    parser.add_argument('--verbose', type=int, default=1)
    parser.add_argument('--csv', type=str, default=None)
    parser.add_argument('--delta-sweep', action='store_true')
    parser.add_argument('--flag-new', action='store_true')

    # v5 flags
    parser.add_argument('--exhaustive', action='store_true',
                        help='dense (rho, delta) grid, operator-level '
                             'exact criterion, quantify negative results')
    parser.add_argument('--rho-fine', type=int, default=RHO_FINE_STEPS,
                        help=f'rho grid points for --exhaustive '
                             f'(default {RHO_FINE_STEPS})')
    parser.add_argument('--delta-fine', type=int, default=DELTA_FINE_STEPS,
                        help=f'delta grid points for --exhaustive '
                             f'(default {DELTA_FINE_STEPS})')
    parser.add_argument('--report', type=str, default=None,
                        help='text file for exhaustive-mode report')

    args, _ = parser.parse_known_args()

    if args.k_all:
        k_list = K_SET
    elif args.k:
        k_list = [int(x) for x in args.k.split(',')]
    else:
        k_list = K_SET

    # ---- exhaustive mode ----
    if args.exhaustive:
        results, _, _ = exhaustive_sweep(
            k_list,
            rho_steps=args.rho_fine,
            delta_steps=args.delta_fine,
            N_max=args.N,
        )
        print_exhaustive_report(results, k_list, out_txt=args.report)
        return

    # ---- v4 delta-sweep mode ----
    if args.delta_sweep:
        for (delta, label) in DELTA_SWEEP:
            print("\n" + "="*70)
            print(f"  DELTA SWEEP : delta = {label}  "
                  f"(numeric = {delta:.10f})")
            print("="*70)
            alpha, beta = 0.0, delta
            beta = snap_delta(beta)

            records = sweep(k_list, args.rho_min, args.rho_max,
                            args.rho_steps, args.N, alpha, beta,
                            args.pos0, args.spin0, verbose=0)
            print_compact_summary(records, k_list,
                                  skip_endpoints=True,
                                  flag_new=args.flag_new)
            if args.csv:
                out = args.csv.replace('.csv',
                                       f"_{label.replace('/','_')}.csv")
                write_csv(records, out)
                print(f"CSV written to {out}")
        return

    # ---- v4 single-delta mode ----
    raw_delta    = args.alpha + args.beta
    snapped      = snap_delta(raw_delta)
    if abs(snapped - raw_delta) > 1e-12:
        args.beta = snapped - args.alpha

    print("="*70)
    print("EXP032b.py v5  —  Dukes quantum walk revival sweep")
    print("="*70)
    print(f"k values    : {k_list}")
    print(f"rho range   : {args.rho_min} .. {args.rho_max} "
          f"({args.rho_steps} points + Dukes exact values)")
    print(f"steps N     : {args.N}")
    if abs(snapped - raw_delta) > 1e-12:
        print(f"NOTE: delta = {raw_delta} snapped to Dukes exact "
              f"value {snapped}")
    print(f"alpha, beta : {args.alpha}, {args.beta}  "
          f"(delta = {args.alpha + args.beta})")
    print(f"initial     : |{args.pos0}, {args.spin0}>")
    print("="*70)

    records = sweep(k_list, args.rho_min, args.rho_max, args.rho_steps,
                    args.N, args.alpha, args.beta,
                    args.pos0, args.spin0, verbose=args.verbose)

    print_compact_summary(records, k_list,
                          skip_endpoints=True,
                          flag_new=args.flag_new)

    if args.csv:
        write_csv(records, args.csv)
        print(f"\nCSV written to {args.csv}")

if __name__ == "__main__":
    main()

Overwriting EXP032b.py


In [ ]:
!python EXP032b.py --verbose 0 --rho-steps 201 --N 60 --csv dukes_scan.csv

EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 20, 24]
rho range   : 0.0 .. 1.0 (201 points + Dukes exact values)
steps N     : 60
alpha, beta : 0.0, 0.0  (delta = 0.0)
initial     : |0, up>

######################################################################
#  COMPACT SUMMARY — FULL revivals (interior rho only)
######################################################################

k = 2
  (continuous family: all rho in [0,1] give N1 = 2 at delta = 0)

k = 3
             rho    N1                 harmonics
        0.333333    12           24, 36, ..., 60
        0.460655    10           20, 30, ..., 60
        0.666667     8           16, 24, ..., 56

k = 4
             rho    N1                 harmonics
        0.250000    12           24, 36, ..., 60
        0.345492    10           20, 30, ..., 60
        0.500000     8           16, 24, ..., 56
        0.750000     6           12, 18, ..., 60
        0.904

run delta sweep

In [ ]:
!python EXP032b.py --delta-sweep --csv sweep.csv


  DELTA SWEEP : delta = 0  (numeric = 0.0000000000)

######################################################################
#  COMPACT SUMMARY — FULL revivals (interior rho only)
######################################################################

k = 2
  (continuous family: all rho in [0,1] give N1 = 2 at delta = 0)

k = 3
             rho    N1                 harmonics
        0.333333    12           24, 36, ..., 60
        0.460655    10           20, 30, ..., 60
        0.666667     8           16, 24, ..., 56

k = 4
             rho    N1                 harmonics
        0.250000    12           24, 36, ..., 60
        0.345492    10           20, 30, ..., 60
        0.500000     8           16, 24, ..., 56
        0.750000     6           12, 18, ..., 60
        0.904508    10           20, 30, ..., 60

k = 5
             rho    N1                 harmonics
        0.276393    60                         -
        0.723607    60                         -

k = 6
            

In [ ]:
from google.colab import files
files.download('dukes_scan.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!python EXP032b.py --k 4 --beta 1.5707963 --N 20 --verbose 0 --csv dukes_scan_k4_pi2.csv

EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [4]
rho range   : 0.0 .. 1.0 (201 points + Dukes exact values)
steps N     : 20
NOTE: delta = 1.5707963 snapped to Dukes exact value 1.5707963267948966
alpha, beta : 0.0, 1.5707963267948966  (delta = 1.5707963267948966)
initial     : |0, up>

######################################################################
#  COMPACT SUMMARY — FULL revivals (interior rho only)
######################################################################

k = 4
             rho    N1                 harmonics
        0.133975    12                         -
        0.500000     6                    12, 18

CSV written to dukes_scan_k4_pi2.csv


# Now analyse results and plot

In [ ]:
%%writefile analyse_dukes_scan.py
#!/usr/bin/env python3
# ============================================================
#  analyse_dukes_scan.py
#
#  Reads dukes_scan.csv (produced by EXP032b.py) and:
#    1. Prints every revival, tagged as "Table N" or "NEW".
#    2. Plots N1 vs rho for each k, with Dukes tabulated
#       values overlaid as red markers.
#    3. Reports a "new solutions" section listing anything
#       not found in DUKES_EXACT.
#
#  Optionally compares two CSVs at different grid resolutions
#  to detect grid artifacts:  pass --fine dukes_scan_fine.csv
# ============================================================

import csv
import argparse
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from math import sqrt, pi

# ------------------------------------------------------------
#  Dukes reference table (same as in EXP032b.py v4)
#  (k, rho, delta, N_expected, source)
# ------------------------------------------------------------

DUKES_EXACT = [
    (2, 0.5, 0.0, 2, "Table 2"),
        (3, 2/3,              0.0,        8,  "Table 3"),
    (3, (5 - sqrt(5))/6,  0.0,        10, "Table 3"),
    (3, 1/3,              0.0,        12, "Table 3"),
    (3, 1/3,              2*pi/3,     12, "Table 3"),
    (3, 1/3,              4*pi/3,     12, "Table 3"),
    # k=6 shares the solution set of k=3  (Dukes §4.3)
    (6, 2/3,              0.0,        8,  "Table 3 (k=3<->6)"),
    (6, (5 - sqrt(5))/6,  0.0,        10, "Table 3 (k=3<->6)"),
    (6, 1/3,              0.0,        12, "Table 3 (k=3<->6)"),
    (6, 1/3,              2*pi/3,     12, "Table 3 (k=3<->6)"),
    (6, 1/3,              4*pi/3,     12, "Table 3 (k=3<->6)"),
    (4, 3/4,              0.0,        6,  "Table 4"),
    (4, 1/2,              0.0,        8,  "Table 4"),
    (4, 1/2,              pi,         8,  "Table 4"),
    (4, (5 - sqrt(5))/8,  0.0,        10, "Table 4"),
    (4, (5 + sqrt(5))/8,  0.0,        10, "Table 4"),
    (4, 1/4,              0.0,        12, "Table 4"),
    (4, 1/4,              pi,         12, "Table 4"),
    (4, 3/4,              pi,         12, "Table 4"),
    (4, (2 - sqrt(3))/2,  pi/2,       12, "Table 4"),
    (4, (2 - sqrt(3))/2,  3*pi/2,     12, "Table 4"),
    (5,  (5 - sqrt(5))/10, 0.0,     60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 0.0,     60, "Table 5"),
    (5,  (5 - sqrt(5))/10, 2*pi/5,  60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 2*pi/5,  60, "Table 5"),
    (5,  (5 - sqrt(5))/10, 4*pi/5,  60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 4*pi/5,  60, "Table 5"),
    (5,  (5 - sqrt(5))/10, 6*pi/5,  60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 6*pi/5,  60, "Table 5"),
    (5,  (5 - sqrt(5))/10, 8*pi/5,  60, "Table 5"),
    (5,  (5 + sqrt(5))/10, 8*pi/5,  60, "Table 5"),
    (10, (5 - sqrt(5))/10, 0.0,     60, "Table 5"),
    (10, (5 + sqrt(5))/10, 0.0,     60, "Table 5"),
    (10, (5 - sqrt(5))/10, 2*pi/5,  60, "Table 5"),
    (10, (5 + sqrt(5))/10, 2*pi/5,  60, "Table 5"),
    (10, (5 - sqrt(5))/10, 4*pi/5,  60, "Table 5"),
    (10, (5 + sqrt(5))/10, 4*pi/5,  60, "Table 5"),
    (10, (5 - sqrt(5))/10, 6*pi/5,  60, "Table 5"),
    (10, (5 + sqrt(5))/10, 6*pi/5,  60, "Table 5"),
    (10, (5 - sqrt(5))/10, 8*pi/5,  60, "Table 5"),
    (10, (5 + sqrt(5))/10, 8*pi/5,  60, "Table 5"),
    (8, 1/2, 0.0,    24, "Table 5"),
    (8, 1/2, pi/2,   24, "Table 5"),
    (8, 1/2, pi,     24, "Table 5"),
    (8, 1/2, 3*pi/2, 24, "Table 5"),
]

# ------------------------------------------------------------
#  CSV reader
# ------------------------------------------------------------

def read_csv(path):
    records = []
    with open(path) as f:
        r = csv.DictReader(f)
        for row in r:
            def parse(s):
                return [int(x) for x in s.split(';')] if s else []
            records.append(dict(
                k=int(row['k']),
                rho=float(row['rho']),
                delta=float(row['delta']),
                full=parse(row['full']),
                pos_only=parse(row['pos_only']),
                coin_only=parse(row['coin_only']),
                both_phase=parse(row['both_phase']),
                coin_triv=parse(row['coin_triv']),
            ))
    return records

# ------------------------------------------------------------
#  Tag revivals against DUKES_EXACT
# ------------------------------------------------------------

def find_source(k, rho, delta, N1,
                rho_tol=1e-4, delta_tol=1e-4, N_tol=1):
    for (kk, rr, dd, NN, src) in DUKES_EXACT:
        if kk != k:
            continue
        if abs(rr - rho) > rho_tol:
            continue
        if abs(dd - delta) > delta_tol:
            continue
        if N1 is not None and abs(NN - N1) > N_tol:
            continue
        return src
    return None

# ------------------------------------------------------------
#  Report
# ------------------------------------------------------------

def report(records, out_txt=None):
    """
    Print the full tagged report.  Optionally write to a text file.
    """
    lines = []
    def p(s=""):
        print(s)
        lines.append(s)

    p("="*72)
    p("Dukes revival analysis")
    p("="*72)

    k_vals = sorted({r['k'] for r in records})
    delta_vals = sorted({r['delta'] for r in records})

    for k in k_vals:
        # Skip k=2 which is a continuous family — handled below.
        if k == 2:
            p(f"\nk = 2")
            p("  (continuous family: all rho in [0,1] give N1 = 2 "
              "at delta = 0)")
            continue

        p(f"\nk = {k}")
        for delta in delta_vals:
            recs = [r for r in records
                    if r['k'] == k and abs(r['delta'] - delta) < 1e-6
                    and r['full']
                    and 1e-6 < r['rho'] < 1 - 1e-6]
            if not recs:
                continue

            p(f"  delta = {delta:.6f}")
            p(f"    {'rho':>12}  {'N1':>4}  {'source':>10}")
            for r in sorted(recs, key=lambda x: x['rho']):
                n1 = r['full'][0]
                src = find_source(k, r['rho'], delta, n1)
                tag = src if src else "NEW"
                p(f"    {r['rho']:12.6f}  {n1:>4}  {tag:>10}")

    # ---- summary of new solutions ----
    p("\n" + "="*72)
        # ---- summary of new solutions ----
    # Only compare the FIRST revival step (the fundamental period).
    # Higher harmonics of a known period are not "new solutions" —
    # they are automatically implied by the known N1.
    p("\n" + "="*72)
    p("Additional solutions (fundamental period not in DUKES_EXACT)")
    p("="*72)
    found_new = False
    for r in records:
        if not r['full']:
            continue
        if r['k'] == 2:
            continue
        if not (1e-6 < r['rho'] < 1 - 1e-6):
            continue
        n1 = r['full'][0]     # fundamental period only
        if find_source(r['k'], r['rho'], r['delta'], n1) is None:
            p(f"  k={r['k']:2d}  rho={r['rho']:.6f}  "
              f"delta={r['delta']:.6f}  N1={n1}")
            found_new = True
    if not found_new:
        p("  (none)")

    if out_txt:
        with open(out_txt, 'w') as f:
            f.write("\n".join(lines))
        print(f"\nReport written to {out_txt}")

# ------------------------------------------------------------
#  Plot
# ------------------------------------------------------------

def plot(records, out_png='dukes_revivals.png'):
    """
    One subplot per k.  x = rho, y = N1 (first revival step).
    Dukes tabulated values are overlaid as red squares.
    """
    k_vals = sorted({r['k'] for r in records if r['k'] != 2})
    n_k = len(k_vals)
    n_cols = 4
    n_rows = (n_k + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(4*n_cols, 3*n_rows),
                             squeeze=False)

    for i, k in enumerate(k_vals):
        ax = axes[i // n_cols][i % n_cols]
        recs = [r for r in records
                if r['k'] == k and r['full']
                and 1e-6 < r['rho'] < 1 - 1e-6]
        if recs:
            xs = [r['rho'] for r in recs]
            ys = [r['full'][0] for r in recs]
            ax.scatter(xs, ys, s=20, c='steelblue',
                       label='detected')

        # Overlay Dukes table entries for this k
        dukes = [(rr, NN, dd) for (kk, rr, dd, NN, _) in DUKES_EXACT
                 if kk == k]
        if dukes:
            dx = [rr for (rr, _, _) in dukes]
            dy = [NN for (_, NN, _) in dukes]
            ax.scatter(dx, dy, s=60, marker='s', facecolors='none',
                       edgecolors='red', linewidths=1.5,
                       label="Dukes' tables")

        ax.set_title(f"k = {k}")
        ax.set_xlabel(r"$\rho$")
        ax.set_ylabel(r"$N_1$")
        ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend(fontsize=7)

    # Hide unused axes
    for j in range(n_k, n_rows*n_cols):
        axes[j // n_cols][j % n_cols].axis('off')

    fig.tight_layout()
    fig.savefig(out_png, dpi=130)
    print(f"\nPlot written to {out_png}")

# ------------------------------------------------------------
#  Optional: stability check across two CSVs
# ------------------------------------------------------------

def stability_check(coarse_path, fine_path,
                    rho_tol=0.01, N_tol=0):
    """
    Compare two CSVs at different rho resolutions.  Report
    which revivals survive both, and which are artifacts of
    the coarse grid.
    """
    coarse = read_csv(coarse_path)
    fine   = read_csv(fine_path)

    def key(r):
        return (r['k'], round(r['rho'], 2), round(r['delta'], 2))

    coarse_rev = [r for r in coarse if r['full']
                  and 1e-6 < r['rho'] < 1 - 1e-6]
    fine_rev   = [r for r in fine   if r['full']
                  and 1e-6 < r['rho'] < 1 - 1e-6]

    print("\n" + "="*72)
    print("Stability check: coarse vs fine grid")
    print("="*72)

    # For each fine revival, find a nearby coarse revival
    stable = []
    unstable = []
    for rf in fine_rev:
        match = None
        for rc in coarse_rev:
            if rc['k'] != rf['k']:
                continue
            if abs(rc['rho'] - rf['rho']) <= rho_tol:
                match = rc
                break
        if match:
            stable.append((rf, match))
        else:
            unstable.append(rf)

    print(f"  Fine-grid revivals: {len(fine_rev)}")
    print(f"  Confirmed on coarse grid: {len(stable)}")
    print(f"  New on fine grid (not on coarse): {len(unstable)}")

    if unstable:
        print("\n  Revivals only on the fine grid (candidates for "
              "narrow solutions):")
        for r in unstable[:20]:
            print(f"    k={r['k']:2d}  rho={r['rho']:.6f}  "
                  f"N1={r['full'][0]}")

# ------------------------------------------------------------
#  Main
# ------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(
        description="Analyse dukes_scan.csv output.")
    parser.add_argument('csv', help='coarse grid CSV')
    parser.add_argument('--fine', default=None,
                        help='fine grid CSV for stability check')
    parser.add_argument('--report', default='dukes_report.txt',
                        help='output text file for the report')
    parser.add_argument('--plot', default='dukes_revivals.png',
                        help='output PNG for the plot')
    parser.add_argument('--no-plot', action='store_true')
    args = parser.parse_args()

    records = read_csv(args.csv)
    report(records, out_txt=args.report)

    if not args.no_plot:
        plot(records, out_png=args.plot)

    if args.fine:
        stability_check(args.csv, args.fine)


if __name__ == "__main__":
    main()

Overwriting analyse_dukes_scan.py


In [ ]:
!python analyse_dukes_scan.py dukes_scan.csv

Dukes revival analysis

k = 2
  (continuous family: all rho in [0,1] give N1 = 2 at delta = 0)

k = 3
  delta = 0.000000
             rho    N1      source
        0.333333    12     Table 3
        0.460655    10     Table 3
        0.666667     8     Table 3

k = 4
  delta = 0.000000
             rho    N1      source
        0.250000    12     Table 4
        0.345492    10     Table 4
        0.500000     8     Table 4
        0.750000     6     Table 4
        0.904508    10     Table 4

k = 5
  delta = 0.000000
             rho    N1      source
        0.276393    60     Table 5
        0.723607    60     Table 5

k = 6
  delta = 0.000000
             rho    N1      source
        0.333333    12  Table 3 (k=3<->6)
        0.460655    10  Table 3 (k=3<->6)
        0.666667     8  Table 3 (k=3<->6)

k = 7

k = 8
  delta = 0.000000
             rho    N1      source
        0.500000    24     Table 5

k = 9

k = 10
  delta = 0.000000
             rho    N1      source
        0.276

Fine scan analysis

In [ ]:
!python EXP032b.py --verbose 0 --rho-steps 2001 --N 60 --csv dukes_scan_fine.csv
!python analyse_dukes_scan.py dukes_scan.csv --fine dukes_scan_fine.csv

EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 20, 24]
rho range   : 0.0 .. 1.0 (2001 points + Dukes exact values)
steps N     : 60
alpha, beta : 0.0, 0.0  (delta = 0.0)
initial     : |0, up>

######################################################################
#  COMPACT SUMMARY — FULL revivals (interior rho only)
######################################################################

k = 2
  (continuous family: all rho in [0,1] give N1 = 2 at delta = 0)

k = 3
             rho    N1                 harmonics
        0.333333    12           24, 36, ..., 60
        0.460655    10           20, 30, ..., 60
        0.666667     8           16, 24, ..., 56

k = 4
             rho    N1                 harmonics
        0.250000    12           24, 36, ..., 60
        0.345492    10           20, 30, ..., 60
        0.500000     8           16, 24, ..., 56
        0.750000     6           12, 18, ..., 60
        0.90

save files from anaysis

In [ ]:
from google.colab import files
files.download('dukes_revivals.png')
files.download('dukes_report.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!python EXP032b.py --k 4 --beta 1.5707963 --N 20 --verbose 0 --csv dukes_scan_k4_pi2.csv
!python analyse_dukes_scan.py dukes_scan_k4_pi2.csv

EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [4]
rho range   : 0.0 .. 1.0 (201 points + Dukes exact values)
steps N     : 20
NOTE: delta = 1.5707963 snapped to Dukes exact value 1.5707963267948966
alpha, beta : 0.0, 1.5707963267948966  (delta = 1.5707963267948966)
initial     : |0, up>

######################################################################
#  COMPACT SUMMARY — FULL revivals (interior rho only)
######################################################################

k = 4
             rho    N1                 harmonics
        0.133975    12                         -
        0.500000     6                    12, 18

CSV written to dukes_scan_k4_pi2.csv
Dukes revival analysis

k = 4
  delta = 1.570796
             rho    N1      source
        0.133975    12     Table 4
        0.500000     6         NEW


Additional solutions (fundamental period not in DUKES_EXACT)
  k= 4  rho=0.500000  delta=1.570796  N1=6

Report written to dukes_report.txt

Plot w

Bring the phase parameter delta  into the mix

In [ ]:
# k=3,6 at delta = 2*pi/3
!python EXP032b.py --k 3,6 --alpha 0 --beta 2.0943951 --N 60 --verbose 0

# k=4 at delta = pi
!python EXP032b.py --k 4 --alpha 0 --beta 3.14159265 --N 60 --verbose 0

# k=4 at delta = pi/2
!python EXP032b.py --k 4 --alpha 0 --beta 1.5707963 --N 60 --verbose 0

# k=5,10 at delta = 2*pi/5
!python EXP032b.py --k 5,10 --alpha 0 --beta 1.2566371 --N 60 --verbose 0

# k=8 at delta = pi/2, pi, 3*pi/2
!python EXP032b.py --k 8 --alpha 0 --beta 1.5707963 --N 60 --verbose 0
!python EXP032b.py --k 8 --alpha 0 --beta 3.14159265 --N 60 --verbose 0
!python EXP032b.py --k 8 --alpha 0 --beta 4.7123889 --N 60 --verbose 0

EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [3, 6]
rho range   : 0.0 .. 1.0 (201 points + Dukes exact values)
steps N     : 60
NOTE: delta = 2.0943951 snapped to Dukes exact value 2.0943951023931953
alpha, beta : 0.0, 2.0943951023931953  (delta = 2.0943951023931953)
initial     : |0, up>

######################################################################
#  COMPACT SUMMARY — FULL revivals (interior rho only)
######################################################################

k = 3
             rho    N1                 harmonics
        0.333333    12           24, 36, ..., 60

k = 6
             rho    N1                 harmonics
        0.333333    12           24, 36, ..., 60
EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [4]
rho range   : 0.0 .. 1.0 (201 points + Dukes exact values)
steps N     : 60
NOTE: delta = 3.14159265 snapped to Dukes exact value 3.141592653589793
alpha, beta : 0.0, 3.141592653589793  (delta = 3.141592653589793)

verify new revival not in Dukes at K=, N=6, rho=1/2 delta = pi/2

Test whether
k=4, δ=π/2 δ=π/2,
ρ=1/2 is state‑specific

In [ ]:
!python EXP032b.py --k 4 --beta 1.5707963 --spin0 down --N 20 --verbose 0
!python EXP032b.py --k 4 --beta 1.5707963 --pos0 1 --spin0 up --N 20 --verbose 0


EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [4]
rho range   : 0.0 .. 1.0 (201 points + Dukes exact values)
steps N     : 20
NOTE: delta = 1.5707963 snapped to Dukes exact value 1.5707963267948966
alpha, beta : 0.0, 1.5707963267948966  (delta = 1.5707963267948966)
initial     : |0, down>

######################################################################
#  COMPACT SUMMARY — FULL revivals (interior rho only)
######################################################################

k = 4
             rho    N1                 harmonics
        0.133975    12                         -
        0.500000     6                    12, 18
EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [4]
rho range   : 0.0 .. 1.0 (201 points + Dukes exact values)
steps N     : 20
NOTE: delta = 1.5707963 snapped to Dukes exact value 1.5707963267948966
alpha, beta : 0.0, 1.5707963267948966  (delta = 1.5707963267948966)
initial     : |1, up>

###############################

Sweep the remaining
k=5,10 deltas

In [ ]:
!python EXP032b.py --k 5,10 --beta 2.5132741 --N 60 --verbose 0    # 4*pi/5
!python EXP032b.py --k 5,10 --beta 3.7699112 --N 60 --verbose 0    # 6*pi/5
!python EXP032b.py --k 5,10 --beta 5.0265482 --N 60 --verbose 0    # 8*pi/5

EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [5, 10]
rho range   : 0.0 .. 1.0 (201 points + Dukes exact values)
steps N     : 60
NOTE: delta = 2.5132741 snapped to Dukes exact value 2.5132741228718345
alpha, beta : 0.0, 2.5132741228718345  (delta = 2.5132741228718345)
initial     : |0, up>

######################################################################
#  COMPACT SUMMARY — FULL revivals (interior rho only)
######################################################################

k = 5
             rho    N1                 harmonics
        0.276393    60                         -
        0.723607    60                         -

k = 10
             rho    N1                 harmonics
        0.276393    60                         -
        0.723607    60                         -
EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [5, 10]
rho range   : 0.0 .. 1.0 (201 points + Dukes exact values)
steps N     : 60
NOTE: delta = 3.7699112 snapped t

Other small tests

In [ ]:
!python EXP032b.py --k 4 --beta 4.7123889 --N 20 --verbose 0
!python EXP032b.py --k 8 --beta 1.5707963 --N 60 --verbose 2
!python EXP032b.py --k 3,6 --beta 1.5707963 --N 40 --verbose 0

EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [4]
rho range   : 0.0 .. 1.0 (201 points + Dukes exact values)
steps N     : 20
NOTE: delta = 4.7123889 snapped to Dukes exact value 4.71238898038469
alpha, beta : 0.0, 4.71238898038469  (delta = 4.71238898038469)
initial     : |0, up>

######################################################################
#  COMPACT SUMMARY — FULL revivals (interior rho only)
######################################################################

k = 4
             rho    N1                 harmonics
        0.133975    12                         -
        0.500000     6                    12, 18
EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [8]
rho range   : 0.0 .. 1.0 (201 points + Dukes exact values)
steps N     : 60
NOTE: delta = 1.5707963 snapped to Dukes exact value 1.5707963267948966
alpha, beta : 0.0, 1.5707963267948966  (delta = 1.5707963267948966)
initial     : |0, up>

  k = 8   delta = 1.570796   (201 rho p

Final verification of Dukes 1D revivals

In [ ]:
# Edit TOL_PHASE to 1e-12 in the file, then:
!python EXP032b.py --k 4 --beta 1.5707963 --N 20 --verbose 0


EXP032b.py v5  —  Dukes quantum walk revival sweep
k values    : [4]
rho range   : 0.0 .. 1.0 (201 points + Dukes exact values)
steps N     : 20
NOTE: delta = 1.5707963 snapped to Dukes exact value 1.5707963267948966
alpha, beta : 0.0, 1.5707963267948966  (delta = 1.5707963267948966)
initial     : |0, up>

######################################################################
#  COMPACT SUMMARY — FULL revivals (interior rho only)
######################################################################

k = 4
             rho    N1                 harmonics
        0.133975    12                         -
        0.500000     6                    12, 18


## Exhaustive search over $(\rho, \delta)$

The grid-search cells above use a 201-point $\rho$ grid, ten fixed
$\delta$ values, and the state-level criterion
`same_up_to_phase(psi, psi0)`. That is enough to confirm the Dukes
entries and to find the $k = 2$ correction, but it is not enough to
support the paper's claim that no revival exists for
$k \notin \{2, 3, 4, 5, 6, 8, 10\}$.

This cell runs `EXP032b.py --exhaustive`, which switches to:

- a dense $(\rho, \delta)$ grid (default $1001 \times 360$, i.e.
  about $3.6 \times 10^5$ points per $k$),
- the operator-level exact criterion $U^N = I$, computed via the
  eigenvalue residual $\max_j |\lambda_j^N - 1|$,
- $N$ from $1$ to $60$ at each grid point.

For each $k$ the output reports:

- every exact revival found, as $(k, \rho, \delta, N_{\text{exact}})$;
- the **worst exact residual over the whole grid**, for $k$ with no
  revival. This is the sensitivity witness: a value of order
  $10^{-1}$ means the search was genuinely sensitive and the
  negative result is meaningful; a value near the $10^{-9}$
  tolerance would mean the grid was flirting with false positives
  and a finer grid might find something.

Runtime is about 5 minutes for the full $k$-set. The report is
written to `exhaustive_report.txt`.

### What to check

- For $k \in \{2, 3, 4, 5, 6, 8, 10\}$: the exact revivals should
  reproduce the `ONE_D_FULL` catalogue from `EXP033.ipynb`.
- For $k \in \{7, 9, 11, 12, 13, 14, 15, 16\}$: no exact revivals,
  with worst residual of order $10^{-1}$.
- The k=2, $\rho=1/2$, $\delta=\pi/2$ correction: `rho=0.500000`
  should appear at `N=24` in the $\delta \approx \pi/2$ block. If
  it appears at `N=6` only, the criterion is projective, not exact,
  and the mode is misconfigured.

In [ ]:
# Quick check on k=7 and k=11 (the negative cases) before the full run.
# Should complete in about a minute. If this works, run Cell C.
!python EXP032b.py --exhaustive --k 7,11 --N 60 \
    --rho-fine 201 --delta-fine 72 \
    --report exhaustive_quick.txt

EXHAUSTIVE (rho, delta) SWEEP — operator-level criterion
  k values          : [7, 11]
  rho grid          : 201 points in [0.0, 1.0]
  delta grid        : 72 points in [0, 2pi)
  N max             : 60
  exact tolerance   : 1.0e-09
  points per k      : 15,276

  k= 7   15270/15276  (100.0%)  exact=52  proj=88  ETA     0s
  k= 7:  candidates=  52  refined=  52  unconfirmed=  0  proj=  88  worst_exact=2.00e+00  (28.4s)
  k=11   15270/15276  (100.0%)  exact=46  proj=82  ETA     0s
  k=11:  candidates=  46  refined=  46  unconfirmed=  0  proj=  82  worst_exact=2.00e+00  (31.9s)

EXHAUSTIVE SWEEP — SUMMARY

k = 7
  grid: 201 rho x 76 delta = 15,276 points
Traceback (most recent call last):
  File "/content/EXP032b.py", line 908, in <module>
    main()
    ~~~~^^
  File "/content/EXP032b.py", line 848, in main
    print_exhaustive_report(results, k_list, out_txt=args.report)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/EXP032b.py", line 606, in print_e

In [ ]:
!python EXP032b.py --exhaustive --k-all --N 60 --report exhaustive_report.txt

EXHAUSTIVE (rho, delta) SWEEP — operator-level criterion
  k values          : [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 20, 24]
  rho grid          : 1001 points in [0.0, 1.0]
  delta grid        : 360 points in [0, 2pi)
  N max             : 60
  exact tolerance   : 1.0e-09
  points per k      : 360,360

  k= 2  360360/360360  (100.0%)  exact=1184  proj=1476  ETA     0s
  k= 2:  candidates=1184  refined=1184  unconfirmed=  0  proj=1476  worst_exact=2.00e+00  (594.3s)
  k= 3  362520/362520  (100.0%)  exact=206  proj=498  ETA     0s
  k= 3:  candidates= 206  refined= 177  unconfirmed= 29  proj= 498  worst_exact=2.00e+00  (613.2s)
  k= 4  362160/362160  (100.0%)  exact=198  proj=490  ETA     0s
  k= 4:  candidates= 198  refined= 188  unconfirmed= 10  proj= 490  worst_exact=2.00e+00  (613.1s)
  k= 5  361800/361800  (100.0%)  exact=198  proj=490  ETA     0s
  k= 5:  candidates= 198  refined= 154  unconfirmed= 44  proj= 490  worst_exact=2.00e+00  (634.4s)
  k= 6  361440/361440  

In [ ]:
from google.colab import files
files.download('exhaustive_report.txt'

In [ ]:
import zipfile
from pathlib import Path

outputs = ["dukes_revivals.png", "dukes_scan_k4_p12.csv",
           "dukes_scan.csv",
           "exhaustive_report.txt",
           "exhaustive_quick.txt",
           "dukes_report.txt"]

with zipfile.ZipFile("EXP032b_v2_outputs.zip", "w",
                     zipfile.ZIP_DEFLATED) as z:
    for f in outputs:
        if Path(f).exists():
            z.write(f)
            print(f"added {f}")
        else:
            print(f"WARNING: {f} not found")

print()
print(f"EXP033b_v2_outputs.zip written "
      f"({Path('EXP032b_v2_outputs.zip').stat().st_size} bytes)")

In [ ]:
ry:
    from google.colab import files
    files.download("EXP032b_v3_outputs.zip")
except ImportError:
    print("Not running in Colab; zip is in the working directory.")

## Re-classification of the exhaustive hits

The exhaustive run above detects all exact revivals on the grid, but
the split between "trivial", "Dukes" and "new" depends on the
`classify_hit` function that was live at the time the run started.
This cell re-classifies the hits with the current `classify_hit`, so
the split is correct regardless of when the `%%writefile` cell was
last executed. It does not re-run the search.

In [ ]:
# Make classify_hit and its dependencies available in this namespace
# (they live in EXP032b.py on disk, written by the %%writefile cell)
import importlib, sys
if 'EXP032b' in sys.modules:
    del sys.modules['EXP032b']
import EXP032b
classify_hit = EXP032b.classify_hit

# Re-classify the in-memory results
print(f"{'k':>3}  {'total':>6}  {'trivial':>7}  {'Dukes':>6}  {'new':>4}")
print("-" * 34)
any_new = False
for k in sorted(results):
    r = results[k]
    trivial = dukes = new = 0
    new_points = []
    for (rho, delta, N) in r['exact_hits']:
        kind = classify_hit(rho, delta, k, N)
        if kind == 'trivial':
            trivial += 1
        elif kind == 'dukes':
            dukes += 1
        else:
            new += 1
            new_points.append((rho, delta, N))
    total = len(r['exact_hits'])
    print(f"{k:>3}  {total:>6}  {trivial:>7}  {dukes:>6}  {new:>4}")
    if new:
        any_new = True
        for (rho, delta, N) in new_points[:8]:
            print(f"       NEW: rho={rho:.6f}  delta={delta:.6f}  N={N}")

print()
if any_new:
    print("Non-trivial new hits exist — inspect them before updating the abstract.")
else:
    print("No new (non-trivial) exact revivals on the grid.")